# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- 1000건마다 `1000.parquet`, `2000.parquet`, ... 형태로 순차 저장
- 중단 후 이어서 크롤링 가능 (기존 파일 자동 감지)
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드

In [ ]:
import pandas as pd
import os
import time
import requests
import urllib3
import ssl
import re
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm

# SSL 경고 무시
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class TLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        ctx.set_ciphers('DEFAULT@SECLEVEL=1')
        kwargs['ssl_context'] = ctx
        return super(TLSAdapter, self).init_poolmanager(*args, **kwargs)

# --- [경로 설정] ---
SAVE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "parquet"))
os.makedirs(SAVE_DIR, exist_ok=True)

# 인증 정보
USER_ID = "925305"
API_KEY = "b462a643571c0dd74c57131f82078b100dec94bd0bfd118fb0e7eb9b95b7182a"
BASE_URL = "https://gelbooru.com/index.php"

def get_info_from_last_file():
    """마지막으로 저장된 파일에서 가장 작은 ID와 파일 번호를 가져옴"""
    files = [f for f in os.listdir(SAVE_DIR) if f.endswith('.parquet')]
    if not files:
        return None, -1
    
    # 파일명 숫자로 정렬
    nums = sorted([int(re.search(r'(\d+)', f).group(1)) for f in files])
    last_num = nums[-1]
    
    # 마지막 파일 로드하여 가장 작은 ID 추출 (다음 호출은 이보다 작아야 함)
    last_file_path = os.path.join(SAVE_DIR, f"{last_num}.parquet")
    df = pd.read_parquet(last_file_path)
    min_id = df['id'].min()
    
    return min_id, last_num

def crawl_gelbooru_id_based():
    session = requests.Session()
    session.mount('https://', TLSAdapter())
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
    })

    # 초기 설정
    last_min_id, last_file_num = get_info_from_last_file()
    current_file_count = last_file_num if last_file_num != -1 else 0
    
    # 전체 개수는 참고용으로만 가져옴
    try:
        init_res = session.get(BASE_URL, params={'page': 'dapi', 's': 'post', 'q': 'index', 'json': 1, 'limit': 1}, verify=False)
        total_posts = int(init_res.json().get('@attributes', {}).get('count', 0))
    except:
        total_posts = "Unknown"

    print(f"서버 전체 데이터: {total_posts}")
    print(f"수집 재개 ID: {last_min_id if last_min_id else '처음부터'}")

    # ID 기반 페이징이므로 total은 예측치로 설정
    pbar = tqdm(desc="Gelbooru Crawling (ID-based)")

    while True:
        current_file_count += 100
        file_path = os.path.join(SAVE_DIR, f"{current_file_count}.parquet")
        
        # 'id:<숫자' 태그를 사용하면 해당 숫자보다 작은 ID의 글을 가져옴 (최신순 기준)
        tag_query = f"id:<{last_min_id}" if last_min_id else ""
        
        params = {
            'page': 'dapi', 's': 'post', 'q': 'index', 'json': 1,
            'limit': 100, 'pid': 0, 'tags': tag_query,
            'api_key': API_KEY, 'user_id': USER_ID
        }
        
        try:
            response = session.get(BASE_URL, params=params, timeout=30, verify=False)
            
            if response.status_code == 200:
                if not response.text.strip() or "Too deep" in response.text:
                    print(f"\n[{current_file_count}] 서버 제한 혹은 빈 응답. 10초 대기...")
                    time.sleep(10)
                    continue

                try:
                    res_data = response.json()
                except:
                    print(f"\n[{current_file_count}] JSON 파싱 에러. 응답 내용: {response.text[:50]}")
                    break

                posts = res_data.get('post', []) if isinstance(res_data, dict) else []
                
                if posts:
                    df = pd.DataFrame(posts)
                    # ID를 숫자로 변환 후 최소값 갱신
                    df['id'] = pd.to_numeric(df['id'])
                    last_min_id = df['id'].min()
                    
                    target_cols = ['id', 'tags', 'sample_url', 'file_url', 'width', 'height']
                    actual_cols = [c for c in target_cols if c in df.columns]
                    
                    df[actual_cols].to_parquet(file_path, engine='pyarrow', index=False)
                    pbar.set_postfix(last_id=last_min_id)
                    pbar.update(1)
                else:
                    print(f"\n[{current_file_count}] 데이터를 모두 수집했습니다.")
                    break
            
            elif response.status_code == 429:
                time.sleep(30)
                continue
            else:
                break
                
        except Exception as e:
            print(f"\n에러: {e}")
            time.sleep(5)
            continue
            
        time.sleep(2.0)

    pbar.close()

if __name__ == "__main__":
    crawl_gelbooru_id_based()

서버 전체 데이터: 약 13210917개 (132110 페이지)
수집 재개 지점: pid 201 (약 20100번 데이터부터)


Gelbooru Crawling:   0%|          | 201/132110 [00:00<?, ?it/s]


[20200] JSON 파싱 불가. 응답 내용 일부: Too deep! Pull it back some. Holy fuck.

[20300] JSON 파싱 불가. 응답 내용 일부: Too deep! Pull it back some. Holy fuck.


KeyboardInterrupt: 